# Object Detection with LTDETRv2

This notebook demonstrates how to use LightlyTrain for object detection with our state-of-the-art LTDETRv2 model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/object_detection.ipynb)

> **Important**: When running on Google Colab make sure to select a GPU runtime for faster processing. You can do this by going to `Runtime` > `Change runtime type` and selecting a GPU hardware accelerator.

## Installation

LightlyTrain can be installed directly via `pip`:

In [ ]:
!pip install lightly-train

> **Important**: LightlyTrain is officially supported on
> - Linux: CPU or CUDA
> - MacOS: CPU only
> - Windows (experimental): CPU or CUDA
>
> We are planning to support MPS for MacOS.
>
> Check the [installation instructions](https://docs.lightly.ai/train/stable/installation.html) for more details on installation.

## Train an LTDETR model

Training your own LTDETR model is straightforward with LightlyTrain.

### Download a dataset

First download a dataset. The dataset must be in YOLO format, see the [documentation](https://docs.lightly.ai/train/stable/object_detection.html#data) for more details. You can use [labelformat](https://github.com/lightly-ai/labelformat) to convert any dataset to the YOLO format.

In [ ]:
!wget https://github.com/lightly-ai/coco128_yolo/releases/download/v0.0.3/object_detection.zip && unzip -q object_detection.zip

The dataset looks like this after the download completes:
```
object_detection
├── config.yaml
├── images
│   ├── train2017
│   │   ├── 000000000009.jpg
│   │   ├── ...
│   │   └── 000000000650.jpg
│   └── val2017
│       ├── 000000000139.jpg
│       ├── ...
│       └── 000000013201.jpg
└── labels
    ├── train2017
    │   ├── 000000000009.txt
    │   ├── ...
    │   └── 000000000650.txt
    └── val2017
        ├── 000000000139.txt
        ├── ...
        └── 000000013201.txt
```

`config.yaml` contains the dataset paths and the 80 COCO class names. Paths in the config file are resolved relative to the config file itself. `data` also accepts a dictionary with the same keys, see the [documentation](https://docs.lightly.ai/train/stable/object_detection.html#data) for details.

### Start training

Next, start the training with the `train_object_detection` function. You only have to specify the output directory, model, and input data. LightlyTrain automatically sets the remaining training parameters and applies image augmentations. Of course you can always customize these settings if needed.

In [ ]:
import lightly_train

lightly_train.train_object_detection(
    out="out/my_experiment",
    model="ltdetrv2-s-coco",
    steps=100,  # Small number of steps for demonstration, default is 266_112.
    batch_size=4,  # Small batch size for demonstration, default is 32.
    data="object_detection/config.yaml",
)

Once the training is complete, the output directory looks like this:
```
out/my_experiment
├── checkpoints
│   ├── best.ckpt
│   └── last.ckpt
├── events.out.tfevents.1764251158.ef9b159fe4b8.273.0
├── exported_models
│   ├── exported_best.pt
│   └── exported_last.pt
└── train.log
```

The best model checkpoint is saved to `out/my_experiment/exported_models/exported_best.pt`.

## Predict with your own LTDETR weights

### Download an example image

Download an example image for inference with the following command:

In [ ]:
!wget -O image.jpg http://images.cocodataset.org/val2017/000000577932.jpg

Then load the model with LightlyTrain's `load_model` function. This will automatically download the model weights and load the model.

### Load the model weights

In [ ]:
model = lightly_train.load_model("out/my_experiment/exported_models/exported_best.pt")

### Predict the objects

Then run `model.predict` on the image. The method accepts file paths, URLs,
PIL Images, or tensors as input.

In [ ]:
results = model.predict("image.jpg")

results["labels"]  # Class labels, tensor of shape (num_boxes,)
results["bboxes"]  # Bounding boxes in (xmin, ymin, xmax, ymax) absolute pixel
# coordinates of the original image. Tensor of shape (num_boxes, 4).
results["scores"]  # Confidence scores, tensor of shape (num_boxes,)

### Visualize the results

Finally, we visualize the image and results to check what objects were detected.

In [ ]:
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.utils import draw_bounding_boxes

image = read_image("image.jpg")
image_with_boxes = draw_bounding_boxes(
    image,
    boxes=results["bboxes"],
    labels=[model.classes[label.item()] for label in results["labels"]],
)
plt.imshow(image_with_boxes.permute(1, 2, 0))
plt.show()

## Benchmark your checkpoint

You can use the `benchmark_object_detection` (in beta) command to measure the inference performance of an object detection model on a validation dataset. 

In [ ]:
result = lightly_train.benchmark_object_detection(
    out="out/my_benchmark",
    dataset_name="coco128",  # Human-readable name shown in the report.
    model="out/my_experiment/exported_models/exported_best.pt",
    data="object_detection/config.yaml",
)

result.print()

The command returns a `BenchmarkResult` instance and writes two files to the out directory:

```
out/my_benchmark
├── benchmark_results.json
└── benchmark_summary.md
```

* `benchmark_results.json`: the full result as JSON.
* `benchmark_summary.md`: a human-readable Markdown report.

The report (also available via `result.to_markdown()`) contains the run configuration, device info, performance metrics, and a throughput & latency table.

## Next Steps

* [Object Detection Documentation](https://docs.lightly.ai/train/stable/object_detection.html): If you want to learn more about object detection with LightlyTrain.
* [Object Detection Model Export](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/object_detection.ipynb): If you want to learn more about export the model to ONNX and TensorRT format.